In [2]:
pip install ortools


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import ortools

In [ ]:
#Variable P=(x,y)  → position of robot
#Domains 𝑥,𝑦 ∈ [0,4] (for 5×5 grid)
#Constraint: 1- P0=(1,1)  2- Last position = (4,4) 3- Diagonal movement only
#            4- Obstacle avoidance 5- No revisiting

#Diagonal move cost = √2

from ortools.sat.python import cp_model

def solve_robot():
    model = cp_model.CpModel()

    size = 5
    path_len = 5

    x = [model.NewIntVar(0, size-1, f'x{i}') for i in range(path_len)]
    y = [model.NewIntVar(0, size-1, f'y{i}') for i in range(path_len)]

    model.Add(x[0] == 0)
    model.Add(y[0] == 0)

    model.Add(x[path_len-1] == 4)
    model.Add(y[path_len-1] == 4)

    grid = [
        [1,1,1,1,1],
        [1,1,1,0,1],
        [1,1,1,1,1],
        [1,0,1,1,1],
        [1,1,1,1,1]
    ]

    for i in range(path_len):
        for r in range(size):
            for c in range(size):
                if grid[r][c] == 0:
                    model.AddForbiddenAssignments([x[i], y[i]], [(r, c)])

    for i in range(path_len - 1):
        a = model.NewIntVar(-1, 1, f'a{i}')
        b = model.NewIntVar(-1, 1, f'b{i}')

        model.Add(a == x[i+1] - x[i])
        model.Add(b == y[i+1] - y[i])

        abs_a = model.NewIntVar(1, 1, f'abs_a{i}')
        abs_b = model.NewIntVar(1, 1, f'abs_b{i}')

        model.AddAbsEquality(abs_a, a)
        model.AddAbsEquality(abs_b, b)

    solver = cp_model.CpSolver()
    status = solver.Solve(model)

    if status == cp_model.FEASIBLE or status == cp_model.OPTIMAL:
        path = []
        for i in range(path_len):
            path.append((solver.Value(x[i]), solver.Value(y[i])))
        return path
    else:
        return "No valid path found"

print("Path:", solve_robot())

Path: [(0, 0), (1, 1), (2, 2), (3, 3), (4, 4)]


In [16]:
from ortools.sat.python import cp_model

def island_perimeter_csp(grid):
    model = cp_model.CpModel()

    rows, cols = len(grid), len(grid[0])
    cell = {}
    for i in range(rows):
        for j in range(cols):
            cell[i, j] = model.NewIntVar(0, 1, f'c{i}{j}')
            model.Add(cell[i, j] == grid[i][j])

    perimeter_terms = []

    for i in range(rows):
        for j in range(cols):
            if grid[i][j] == 1:

                if i == 0:
                    perimeter_terms.append(1)
                else:
                    perimeter_terms.append(1 - cell[i-1, j])

                if i == rows - 1:
                    perimeter_terms.append(1)
                else:
                    perimeter_terms.append(1 - cell[i+1, j])

                if j == 0:
                    perimeter_terms.append(1)
                else:
                    perimeter_terms.append(1 - cell[i, j-1])

                if j == cols - 1:
                    perimeter_terms.append(1)
                else:
                    perimeter_terms.append(1 - cell[i, j+1])

    perimeter = model.NewIntVar(0, rows * cols * 4, "perimeter")
    model.Add(perimeter == sum(perimeter_terms))

    solver = cp_model.CpSolver()
    solver.Solve(model)

    return solver.Value(perimeter)

grid = [
    [1,1,0,0,0],
    [1,1,0,1,1],
    [0,0,0,1,1],
    [0,1,1,0,0],
    [0,1,1,0,0]
]

print("Largest continuous landmass Perimeter:", island_perimeter_csp(grid))
                

Largest continuous landmass Perimeter: 24


In [17]:
#Variable route[i]: city visited at position i
#Domains route[i]∈{0,1,2,...,9}
#Constraint: 1- No revisiting  2- Start and end at same city 3- Valid path btw cities

from ortools.constraint_solver import pywrapcp, routing_enums_pb2

def tsp():
    dist = [
        [0,29,20,21,16,31,100,12,4,31],
        [29,0,15,29,28,40,72,21,29,41],
        [20,15,0,15,14,25,81,9,23,27],
        [21,29,15,0,4,12,92,12,25,13],
        [16,28,14,4,0,16,94,9,20,16],
        [31,40,25,12,16,0,95,24,36,3],
        [100,72,81,92,94,95,0,90,101,99],
        [12,21,9,12,9,24,90,0,15,25],
        [4,29,23,25,20,36,101,15,0,35],
        [31,41,27,13,16,3,99,25,35,0]
    ]

    manager = pywrapcp.RoutingIndexManager(len(dist), 1, 0)
    routing = pywrapcp.RoutingModel(manager)

    def distance(from_i, to_i):
        return dist[manager.IndexToNode(from_i)][manager.IndexToNode(to_i)]

    routing.SetArcCostEvaluatorOfAllVehicles(
        routing.RegisterTransitCallback(distance)
    )

    params = pywrapcp.DefaultRoutingSearchParameters()
    params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC

    solution = routing.SolveWithParameters(params)
    route = []
    index = routing.Start(0)

    while not routing.IsEnd(index):
        route.append(manager.IndexToNode(index))
        index = solution.Value(routing.NextVar(index))

    route.append(manager.IndexToNode(index))
    return route


print("Optimal Route:", tsp())

Optimal Route: [0, 8, 7, 2, 1, 6, 5, 9, 3, 4, 0]
